In [1]:
import pandas as pd
import numpy as np
import optuna
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OrdinalEncoder, TargetEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import ElasticNet
from sklearn.neural_network import MLPRegressor

# Load data (Sesuaikan path file jika perlu)
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [2]:
df = pd.concat([train, test], ignore_index=True)
df

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500.0
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500.0
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500.0
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000.0
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2914,2915,160,RM,21.0,1936,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,6,2006,WD,Normal,NaN
2915,2916,160,RM,21.0,1894,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,4,2006,WD,Abnorml,NaN
2916,2917,20,RL,160.0,20000,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2006,WD,Abnorml,NaN
2917,2918,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,MnPrv,Shed,700,7,2006,WD,Normal,NaN


In [3]:
# 1. KATEGORIKAL ORDINAL
qual_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 
             'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']

for col in qual_cols:
    if col != 'KitchenQual':
        df[col] = df[col].fillna('None')

df['KitchenQual'] = df['KitchenQual'].fillna(train['KitchenQual'].mode()[0])

qual_mapping = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
for col in qual_cols:
    df[col] = df[col].map(qual_mapping)

df['Functional'] = df['Functional'].fillna('Typ').map({'Sal': 1, 'Sev': 2, 'Maj2': 3, 'Maj1': 4, 'Mod': 5, 'Min2': 6, 'Min1': 7, 'Typ': 8})
df['LandSlope'] = df['LandSlope'].fillna('Gtl').map({'Sev': 1, 'Mod': 2, 'Gtl': 3})

In [4]:
# 2. KATEGORIKAL NOMINAL
nominal_cols = [
    'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 
    'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 
    'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'Foundation', 'BsmtExposure', 
    'BsmtFinType1', 'BsmtFinType2', 'Heating', 'CentralAir', 'Electrical', 'GarageType', 
    'GarageFinish', 'PavedDrive', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition'
]

none_fill_cols = ['Alley', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'GarageType', 'GarageFinish', 'Fence', 'MiscFeature', 'MasVnrType']

for col in nominal_cols:
    if col in none_fill_cols:
        df[col] = df[col].fillna('None')
    else:
        df[col] = df[col].fillna(train[col].mode()[0])

for col in nominal_cols:
    df[col] = df[col].astype('category')

In [5]:
# 3. NUMERIK
num_cols_to_zero = ['MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'GarageCars', 'GarageArea']
for col in num_cols_to_zero:
    df[col] = df[col].fillna(0)

df['GarageYrBlt'] = df['GarageYrBlt'].fillna(df['YearBuilt'])

frontage_median = train.groupby('Neighborhood')['LotFrontage'].median()
df['LotFrontage'] = df['LotFrontage'].fillna(df['Neighborhood'].map(frontage_median))
df['LotFrontage'] = df['LotFrontage'].fillna(train['LotFrontage'].median())

In [6]:
# 4. TARGET TRANSFORMATION
df['SalePrice'] = np.log1p(df['SalePrice'])

print("Preprocessing selesai. Data siap di-split.")

Preprocessing selesai. Data siap di-split.


In [7]:
test_clean = df[df['SalePrice'].isnull()].copy()

X_test_final = test_clean.drop(columns=['SalePrice', 'Id'])

In [8]:
train_clean = df[df['SalePrice'].notnull()].copy()

In [9]:
# Target & Features
target_col = 'SalePrice'
X = train_clean.drop(columns=['Id', target_col])

# Log1p Transform untuk target agar MSE/RMSE menjadi ekuivalen dengan MSLE/RMSLE
y = np.log1p(train_clean[target_col])

# Pisahkan deteksi kolom kategorikal dan numerikal
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object', 'category']).columns.tolist()

# Imputasi awal agar model dan encoder tidak gagal saat mendeteksi NaN
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='constant', fill_value='Missing')

X_base = X.copy()
X_base[num_cols] = num_imputer.fit_transform(X_base[num_cols])
X_base[cat_cols] = cat_imputer.fit_transform(X_base[cat_cols])

# Global dictionary penampung hasil
oof_scores = {}
best_params = {}

In [10]:
# Format data agar komputasi Tree-Based berjalan native tanpa ekspansi fitur (OHE)

# Ordinal Encoder mengubah string menjadi angka
ordinal_enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_tree_encoded = X_base.copy()
X_tree_encoded[cat_cols] = ordinal_enc.fit_transform(X_tree_encoded[cat_cols])

# 1. Format CatBoost (memerlukan tipe integer untuk cat_features)
X_cb = X_tree_encoded.copy()
X_cb[cat_cols] = X_cb[cat_cols].astype(int)

# 2. Format LGBM & XGBoost (menggunakan internal categorical handling dengan tipe data 'category')
X_tree_cat = X_tree_encoded.copy()
X_tree_cat[cat_cols] = X_tree_cat[cat_cols].astype('category')

In [11]:
# Setup 5-Fold CV (Stabil untuk regresi)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [12]:
def objective_cb(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 200, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
        'random_seed': 42,
        'verbose': False
    }
    
    cv_scores = []
    for train_idx, val_idx in kf.split(X_cb):
        X_tr, y_tr = X_cb.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X_cb.iloc[val_idx], y.iloc[val_idx]
        
        # Injeksi cat_features
        model = CatBoostRegressor(**params, cat_features=cat_cols)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=50, verbose=False)
        
        preds = model.predict(X_va)
        cv_scores.append(np.sqrt(mean_squared_error(y_va, preds)))
        
    return np.mean(cv_scores)

study_cb = optuna.create_study(direction='minimize')
study_cb.optimize(objective_cb, n_trials=10) # Naikkan n_trials di run akhir

oof_scores['CatBoost'] = study_cb.best_value
best_params['CatBoost'] = study_cb.best_params

[I 2026-07-09 16:21:08,854] A new study created in memory with name: no-name-59d2eed4-0b00-4b6f-bdab-85338a21d6e7
[I 2026-07-09 16:28:07,183] Trial 0 finished with value: 0.019899386953320972 and parameters: {'iterations': 403, 'learning_rate': 0.002175267773465826, 'depth': 9, 'l2_leaf_reg': 4.960450278452433}. Best is trial 0 with value: 0.019899386953320972.
[I 2026-07-09 16:35:10,562] Trial 1 finished with value: 0.010167500112248448 and parameters: {'iterations': 440, 'learning_rate': 0.014956206617426653, 'depth': 8, 'l2_leaf_reg': 0.03502023298118703}. Best is trial 1 with value: 0.010167500112248448.
[I 2026-07-09 16:40:52,839] Trial 2 finished with value: 0.01755202423149666 and parameters: {'iterations': 739, 'learning_rate': 0.0014831578681294745, 'depth': 5, 'l2_leaf_reg': 0.05892315126509849}. Best is trial 1 with value: 0.010167500112248448.
[I 2026-07-09 16:44:40,975] Trial 3 finished with value: 0.01010197072338166 and parameters: {'iterations': 645, 'learning_rate': 0.

CatBoostError: (Error 32: The process cannot access the file because it is being used by another process.) util/system/file.cpp:936: can't open "catboost_info\\time_left.tsv" with mode WrOnly|CreateAlways|Seq (0x00000034)

In [ ]:
def objective_lgbm(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'random_state': 42,
        'verbose': -1
    }
    
    cv_scores = []
    for train_idx, val_idx in kf.split(X_tree_cat):
        X_tr, y_tr = X_tree_cat.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X_tree_cat.iloc[val_idx], y.iloc[val_idx]
        
        # Kategori ditangani otomatis berkat tipe data 'category'
        model = LGBMRegressor(**params)
        model.fit(X_tr, y_tr)
        
        preds = model.predict(X_va)
        cv_scores.append(np.sqrt(mean_squared_error(y_va, preds)))
        
    return np.mean(cv_scores)

study_lgbm = optuna.create_study(direction='minimize')
study_lgbm.optimize(objective_lgbm, n_trials=10)

oof_scores['LightGBM'] = study_lgbm.best_value
best_params['LightGBM'] = study_lgbm.best_params

In [ ]:
def objective_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'enable_categorical': True, # Wajib agar kolom 'category' terbaca
        'tree_method': 'hist',      # Algoritma histogram yang mendukung kategorikal
        'random_state': 42
    }
    
    cv_scores = []
    for train_idx, val_idx in kf.split(X_tree_cat):
        X_tr, y_tr = X_tree_cat.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X_tree_cat.iloc[val_idx], y.iloc[val_idx]
        
        model = XGBRegressor(**params)
        model.fit(X_tr, y_tr, verbose=False)
        
        preds = model.predict(X_va)
        cv_scores.append(np.sqrt(mean_squared_error(y_va, preds)))
        
    return np.mean(cv_scores)

study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=10)

oof_scores['XGBoost'] = study_xgb.best_value
best_params['XGBoost'] = study_xgb.best_params

In [ ]:
def objective_en(trial):
    params = {
        'alpha': trial.suggest_float('alpha', 1e-4, 10.0, log=True),
        'l1_ratio': trial.suggest_float('l1_ratio', 0.0, 1.0),
        'random_state': 42,
        'max_iter': 2000
    }
    
    cv_scores = []
    # Loop menggunakan X_base (string kategorikal awal) karena akan diurus oleh TargetEncoder di Pipeline
    for train_idx, val_idx in kf.split(X_base):
        X_tr, y_tr = X_base.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X_base.iloc[val_idx], y.iloc[val_idx]
        
        # Pipeline: TargetEncoding -> StandardScaler
        prep = ColumnTransformer([
            ('cat_enc', TargetEncoder(target_type='continuous'), cat_cols)
        ], remainder='passthrough')
        
        pipe = Pipeline([
            ('prep', prep),
            ('scaler', StandardScaler()),
            ('model', ElasticNet(**params))
        ])
        
        pipe.fit(X_tr, y_tr)
        preds = pipe.predict(X_va)
        cv_scores.append(np.sqrt(mean_squared_error(y_va, preds)))
        
    return np.mean(cv_scores)

study_en = optuna.create_study(direction='minimize')
study_en.optimize(objective_en, n_trials=10)

oof_scores['ElasticNet'] = study_en.best_value
best_params['ElasticNet'] = study_en.best_params

In [ ]:
def objective_mlp(trial):
    params = {
        'hidden_layer_sizes': trial.suggest_categorical('hidden_layer_sizes', [(64,), (128,), (64, 32), (128, 64)]),
        'alpha': trial.suggest_float('alpha', 1e-4, 1.0, log=True),
        'learning_rate_init': trial.suggest_float('learning_rate_init', 1e-4, 0.01, log=True),
        'random_state': 42,
        'max_iter': 1000,
        'early_stopping': True
    }
    
    cv_scores = []
    for train_idx, val_idx in kf.split(X_base):
        X_tr, y_tr = X_base.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X_base.iloc[val_idx], y.iloc[val_idx]
        
        prep = ColumnTransformer([
            ('cat_enc', TargetEncoder(target_type='continuous'), cat_cols)
        ], remainder='passthrough')
        
        pipe = Pipeline([
            ('prep', prep),
            ('scaler', StandardScaler()),
            ('model', MLPRegressor(**params))
        ])
        
        pipe.fit(X_tr, y_tr)
        preds = pipe.predict(X_va)
        cv_scores.append(np.sqrt(mean_squared_error(y_va, preds)))
        
    return np.mean(cv_scores)

study_mlp = optuna.create_study(direction='minimize')
study_mlp.optimize(objective_mlp, n_trials=10)

oof_scores['MLPRegressor'] = study_mlp.best_value
best_params['MLPRegressor'] = study_mlp.best_params

In [ ]:
# Kompilasi hasil validasi OOF ke dalam tabel urut
eval_df = pd.DataFrame(list(oof_scores.items()), columns=['Model', 'OOF_RMSLE'])
eval_df = eval_df.sort_values(by='OOF_RMSLE', ascending=True).reset_index(drop=True)

print("=== Peringkat Evaluasi OOF RMSLE (Skor terkecil = Terbaik) ===")
display(eval_df)

In [ ]:
# Print config final agar gampang disalin/dicatat
print("=== Parameter Terbaik Hasil Tuning Optuna ===")
for model_name, params in best_params.items():
    print(f"\n[{model_name}]")
    for k, v in params.items():
        print(f"  {k}: {v}")